# Embedding visualization for tests

- **Goal**: Check that the embedding generated by the test code makes sense.
- **Authors**: JGY <jyoung22@uvm.edu>, Ariane Lizotte <ariane.lizotte.2@ulaval.ca> 
- **Last modified**: April 2025
- **License**: CC BY 4.0

# Imports

In [ ]:
import pickle
import json

import matplotlib.pyplot as plt
import numpy as np
import jax.numpy as jnp
import networkx as nx

import sys
sys.path.append("../")
from mdl_hyperbolic.models import hyperbolic_model, connection_probs, expand_positions, distance_matrix, hyperbolic_loglikelihood

In [ ]:
# load embedding
with open("../.output/hyperbolic_model.pkl", "rb") as f:
    optimized_embedding = pickle.load(f)

In [ ]:
g = optimized_embedding['graph']
A = nx.linalg.adjacency_matrix(g).todense()
inferred_x = optimized_embedding['x']
inferred_r = optimized_embedding['r']
inferred_beta = optimized_embedding['beta']
num_dim = inferred_x.shape[1]

# Coordinate conversion

The embedding finds positions in $\mathbb{H}^D$ (with $D=2$ in our test), so we need to coordinate to polar coordinates for plotting.

In [ ]:
def plot_with_geodesics(x, g, num_pts=100, **edge_kws):
    """
    x: (n,3) hyperboloid coords
    g: networkx.Graph with nodes 0…n-1
    num_pts: points per geodesic
    edge_kws: passed to ax.plot for styling edges
    """
    # node positions in (r,θ)
    r_nodes     = jnp.arccosh(x[:, -1])
    theta_nodes = jnp.arctan2(x[:,1], x[:,0])

    fig, ax = plt.subplots(1,1,figsize=(8,8), subplot_kw={'projection':'polar'})

    # draw each edge as a sampled geodesic
    for i,j in g.edges():
        u, v = x[i], x[j]

        # hyperbolic distance between u,v
        mip  = jnp.dot(u[:-1], v[:-1]) - u[-1]*v[-1]
        d    = jnp.arccosh(-mip)

        # geodesic interpolation on hyperboloid
        t = jnp.linspace(0,1, num_pts)
        a = jnp.sinh((1 - t)*d) / jnp.sinh(d)
        b = jnp.sinh(    t *d) / jnp.sinh(d)
        curve = a[:,None]*u[None,:] + b[:,None]*v[None,:]  # (num_pts,3)

        # project each intermediate to (r,θ)
        r_curve     = jnp.arccosh(curve[:,-1])
        theta_curve = jnp.arctan2(curve[:,1], curve[:,0])

        ax.plot(np.asarray(theta_curve),
                np.asarray(r_curve),
                **edge_kws)
    ax.scatter(np.asarray(theta_nodes),
               np.asarray(r_nodes),
               s=10, alpha=0.8, color='C0')

    ax.set_rticks([])
    ax.set_xticks([])
    ax.set_title("Hyperbolic Embedding with Geodesic Edges")
    plt.show()
# x is your (n,3) array; g your networkx graph
plot_with_geodesics(expand_positions(inferred_x[:, :2]), g,
    num_pts=50,
    lw=0.5,           # line‐width
    color='gray',     # edge color
    alpha=0.7         # transparency
)

In [ ]:
np.set_printoptions(suppress=True) # Display only in decimal
#inferred_x

## Log-likelihood vs beta and r

In [ ]:
losses = np.loadtxt("../.output/loss.txt")
betas = np.loadtxt("../.output/beta.txt")
# rs = np.loadtxt("../r.txt")
rs = np.full_like(betas, inferred_r)
iterations = 100*np.arange(len(losses))

#last_iter = 100*np.arange(len(losses))[~np.isnan(losses)][-1]
#last_iter += 100
snapshots = [100, 5000, 15000, (len(losses)-1)*100]

from mpl_toolkits.axes_grid1 import host_subplot
import mpl_toolkits.axisartist as AA
import matplotlib.pyplot as plt

host = host_subplot(111, axes_class=AA.Axes)
plt.subplots_adjust(right=0.75)

par1 = host.twinx()
par2 = host.twinx()

offset = 60
new_fixed_axis = par2.get_grid_helper().new_fixed_axis
par2.axis["right"] = new_fixed_axis(loc="right",
                                    axes=par2,
                                    offset=(offset, 0))

par1.axis["right"].toggle(all=True)
par2.axis["right"].toggle(all=True)

host.set_xlabel("Iterations")
host.set_ylabel("-log posterior")
par1.set_ylabel("beta")
par2.set_ylabel("r")

p1, = host.plot(iterations, losses, label="-log posterior")
for snapshot in snapshots:
    host.axvline(snapshot, color="k", ls="--")

p2, = par1.plot(iterations, betas, label="beta", ls="--")
p3, = par2.plot(iterations, rs, label="r")

host.legend()

host.axis["left"].label.set_color(p1.get_color())
par1.axis["right"].label.set_color(p2.get_color())
par2.axis["right"].label.set_color(p3.get_color())

plt.draw()
plt.show()

## Inferred conditional connection probabilities
We look at the posterior distribution of distances $p(d(x_u, x_v)|G)$ for $a_{uv}=1$ and $a_{uv}=0$ as histograms of the MLE.

In [ ]:
def plot_cond_connection_prob_dist(ax, A, embedding, beta, r):
    dist = distance_matrix(expand_positions(embedding))
    xvalues = np.linspace(0, np.max(dist), 100)
    ax.plot(xvalues, 1/(1+np.exp(beta*(xvalues - r))), color="k")
    ax2 = ax.twinx()
    ax2.yaxis.set_visible(False)
    triu = np.triu_indices(A.shape[0], k=1)
    A_up = A[triu]
    ax2.hist([np.ravel(dist[triu][A_up==0]), np.ravel(dist[triu][A_up==1])],
             label=["Distances for aij=0", "Distances for aij=1"],
             color=["tab:blue", "tab:orange"], alpha=0.5, density=True, bins=20, stacked=True)
    ax.set_xlabel("distance")
    ax.set_ylabel("connection prob")

def plot_cond_connection_prob(ax, A, embedding, beta, r):
    x_full = expand_positions(embedding)    
    p = connection_probs(x_full, beta, r)
    p = jnp.clip(p, a_min=1e-7, a_max=1 - 1e-7)

    triu = np.triu_indices(A.shape[0], k=1)
    A_up = A[triu]
    ax.hist([np.ravel(p[triu][A_up==0]), np.ravel(p[triu][A_up==1])], label=["aij=0", "aij=1"], density=True, alpha=0.5)
    ax.set_xlabel("Connection prob. | aij")
    ax.yaxis.set_visible(False)

In [ ]:
fig, axes = plt.subplots(2, len(snapshots), figsize=(13, 6), sharey="row", tight_layout=True)
for i, snapshot in enumerate(snapshots):
    it = snapshot
    embedding = np.load(f"../.output/embedding_{it}.npy")
    beta = betas[int(it/100)]
    r = rs[int(it/100)]
    
    axes[0, i].set_title(f"it={snapshot}")
    plot_cond_connection_prob(axes[0, i], A, embedding, beta, r)
    plot_cond_connection_prob_dist(axes[1, i], A, embedding, beta, r)
    
    if i != 0:
        axes[1, i].set_ylabel("")

In [ ]:
from dmercator_api import embed_dmercator

loss_function = lambda x, beta, r: -hyperbolic_loglikelihood(A, expand_positions(x), beta, r)

fig, axes = plt.subplots(2, 3, figsize=(10, 4), tight_layout=True)
mercator_model = embed_dmercator("../.output/tmp_edgelist.txt", num_dim)
axes[0, 0].set_title(f"D-Mercator {loss_function(*mercator_model):.2f}")
plot_cond_connection_prob(axes[0, 0], A, *mercator_model)
plot_cond_connection_prob_dist(axes[1, 0], A, *mercator_model)
axes[0, 0].legend()

inferred_model = (inferred_x, inferred_beta, inferred_r)
axes[0, 1].set_title(f"Gradient descent {loss_function(*inferred_model):.2f}")
plot_cond_connection_prob(axes[0, 1], A, *inferred_model)
plot_cond_connection_prob_dist(axes[1, 1], A, *inferred_model)

with open("../ground_truth.json", "r") as file_stream:
    ground_truth = json.load(file_stream)

ground_truth_model = (jnp.array(ground_truth["positions"]), ground_truth["beta"], ground_truth["r"])
axes[0, 2].set_title(f"Ground truth {loss_function(*ground_truth_model):.2f}")
plot_cond_connection_prob(axes[0, 2], A, *ground_truth_model)
plot_cond_connection_prob_dist(axes[1, 2], A, *ground_truth_model)
plt.show()